# Step 4 — Reusable data-quality helpers

This notebook tests the shared validation and PII-masking helpers before they are called from Airflow. The source CSV files are read only; no input file is changed.

Flow: load data → run six quality dimensions → inspect `dq_results` → inspect quarantine candidates → test masking → save local outputs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Add the repository's include directory so the notebook uses the same helpers as Airflow.
INCLUDE_DIR = Path('include')
sys.path.insert(0, str(INCLUDE_DIR.resolve()))

from data_quality import run_quality_checks
from masking import mask_email, mask_phone

DATA_DIR = Path('include/data/nammamart')
FILES = {
    'orders': 'orders.csv',
    'orders_increment': 'orders_2026-10-01_increment.csv',
    'payments': 'payments.csv',
    'customers': 'customers.csv',
    'products': 'products.csv',
    'stores': 'stores.csv',
}

# Keeping the file-to-DataFrame mapping explicit makes the later DAG calls easy to follow.
data = {name: pd.read_csv(DATA_DIR / filename) for name, filename in FILES.items()}
for name, frame in data.items():
    print(f'{name:18} {len(frame):5,} rows loaded')

In [ ]:
# Run all shared validation rules with the same identifiers that will be stored in DuckDB.
issues, dq_results = run_quality_checks(
    data,
    run_id='step4-local-test',
    dag_id='nammamart_quality_helpers_test',
)

print(f'Issue instances: {len(issues):,}')
print(f'Quality checks: {len(dq_results):,}')
print(f'Failed checks: {(dq_results["status"] == "FAIL").sum():,}')
display(dq_results[['check_name', 'dimension', 'severity', 'rows_checked', 'rows_failed', 'status']])

In [ ]:
# Summarise failures by file, dimension and severity so the ETL DAG can route them.
display(issues.groupby(['file', 'quality_dimension', 'severity']).size().rename('issue_rows').reset_index())

# Critical rows are quarantine candidates; warnings remain loadable but are logged.
quarantine_candidates = issues[issues['severity'] == 'Critical']
warning_rows = issues[issues['severity'] == 'Warning']
print(f'Critical quarantine candidates: {len(quarantine_candidates):,}')
print(f'Warning rows to log: {len(warning_rows):,}')
display(issues.head(20))

In [ ]:
# Verify that personal data is masked before it can enter reporting tables.
mask_examples = pd.DataFrame({
    'original_email': ['priya.hegde199@example.com', 'invalid-email'],
    'masked_email': [mask_email('priya.hegde199@example.com'), mask_email('invalid-email')],
    'original_phone': ['9774349021', '123'],
    'masked_phone': [mask_phone('9774349021'), mask_phone('123')],
})
display(mask_examples)

# These files mirror the future dq_results and quarantine inputs to the pipeline.
Path('assignment').mkdir(exist_ok=True)
issues.to_csv('assignment/data_issues_log_step4.csv', index=False)
dq_results.to_csv('assignment/dq_results_step4.csv', index=False)
print('Wrote assignment/data_issues_log_step4.csv')
print('Wrote assignment/dq_results_step4.csv')